<a href="https://colab.research.google.com/github/Ravindra1972/Anaytics-in-finance-using-Python/blob/main/Spread_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Analyze liquidity using bid-ask spread .
#Demonstrates how spread varies across stocks .

import numpy as np

# Simulated bid- ask data for different stocks
stocks = {
    'AAPL': {'bid': 175.00, 'ask': 175.01, 'vol': 80_000_000},
    'MSFT': {'bid': 380.50, 'ask': 380.52, 'vol': 25_000_000},
    'PENNY': {'bid': 0.45, 'ask': 0.55, 'vol': 50_000},
}

print(f"{' Stock ':<8} {' Spread ':<8} {' Spread % ':<10} {' Volume ':<12}")
print("-" * 42)
for name, data in stocks.items():
    spread = data['ask'] - data['bid']
    mid = (data['ask'] + data['bid']) / 2
    spread_pct = (spread / mid) * 100
    print(f"{name:<8} ${spread:>7.3f} {spread_pct:>9.3f}% "
          f"{data['vol']:>12,}")

 Stock    Spread   Spread %   Volume     
------------------------------------------
AAPL     $  0.010     0.006%   80,000,000
MSFT     $  0.020     0.005%   25,000,000
PENNY    $  0.100    20.000%       50,000


In [16]:
#Chapter 1: Complete Market Simulation
#---------------------------------------
# Simulates a market with :
#5 - Multiple participants ( hedgers , speculators )
#6 - Random orders
#7 - Order matching
#8 - Spread tracking
import random
import numpy as np

class MarketSimulator:
    """A simple market with order matching."""

    def __init__(self, initial_price=100.0):
        self.price = initial_price
        self.bid = initial_price - 0.05
        self.ask = initial_price + 0.05
        self.trade_history = []
        self.spread_history = []

    def generate_order(self, participant_type):
        """ Generate a random order based on
         participant type."""
        if participant_type == 'hedger':
            # Hedgers buy/sell based on exposure
            side = random.choice(['BUY', 'SELL'])
            size = random.randint(50, 200)
        elif participant_type == 'speculator':
            # Speculators bet on direction
            side = 'BUY' if random.random() > 0.5 \
                else 'SELL'
            size = random.randint(100, 500)
        elif participant_type == 'market_maker':
            # Market makers provide both sides
            return None  # They set the spread
        return {'side': side, 'size': size}

    def execute_trade(self, order):
        """ Execute a trade and update the price."""
        if order is None:
            return
        # Simple price impact model
        impact = order['size'] * 0.001
        if order['side'] == 'BUY':
            trade_price = self.ask
            self.price += impact  # Price goes up
        else:
            trade_price = self.bid
            self.price -= impact  # Price goes down

        # Update bid/ask
        spread = 0.05 + random.uniform(0, 0.05)
        self.bid = self.price - spread / 2
        self.ask = self.price + spread / 2

        # Record
        self.trade_history.append({
            'price': trade_price,
            'side': order['side'],
            'size': order['size']
        })
        self.spread_history.append(
            self.ask - self.bid)
        return trade_price

    def run_simulation(self, n_trades=100):
        """Run a full simulation."""
        participants = ['hedger'] * 3 + \
                       ['speculator'] * 5 + \
                       ['market_maker'] * 2

        for i in range(n_trades):
            p_type = random.choice(participants)
            order = self.generate_order(p_type)
            self.execute_trade(order)

        return self.summarize()

    def summarize(self):
        """ Print market summary statistics."""
        prices = [t['price'] for t in self.trade_history]
        spreads = self.spread_history

        print("=== MARKET SIMULATION SUMMARY === ")
        print(f" Total trades : {len(prices)}")
        print(f" Opening price : ${prices[0]:.2f}")
        print(f" Closing price : ${prices[-1]:.2f}")
        print(f" Return : {(prices[-1] / prices[0] - 1) * 100:.2f}%")
        print(f" High : ${max(prices):.2f}")
        print(f"Low: ${min(prices):.2f}")
        print(f"Avg spread : ${np.mean(spreads):.4f}")
        print(f" Volatility : ${np.std(prices):.2f}")
        return prices

# --- Run it! ---
random.seed(42)
sim = MarketSimulator(initial_price=100.0)
prices = sim.run_simulation(n_trades=200)

=== MARKET SIMULATION SUMMARY === 
 Total trades : 154
 Opening price : $100.05
 Closing price : $98.74
 Return : -1.31%
 High : $102.75
Low: $98.32
Avg spread : $0.0747
 Volatility : $0.85
